# Understanding Agentic AI with Local Ollama LLM

This notebook introduces **Agentic AI** concepts and demonstrates how to build AI agents using a locally-hosted LLM via [Ollama](https://ollama.ai).

## What is Agentic AI?

Agentic AI refers to AI systems that can:
- **Reason** about problems step-by-step
- **Plan** and decompose complex tasks
- **Act** by calling tools and executing code
- **Observe** results and adapt their approach
- **Remember** context across interactions

Unlike simple prompt-response LLMs, agents operate in a loop: **Think → Act → Observe → Repeat** until the task is complete.

## Prerequisites
- Ollama installed and running locally (`ollama serve`)
- A model pulled (e.g., `ollama pull llama3.1`)


## 1. Install and Import Dependencies

Install the required packages for working with Ollama and building agents.

In [ ]:
# Install required packages (uncomment and run if not already installed)
# !pip install ollama langchain langchain-ollama langgraph

In [ ]:
import ollama
import json
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ConversationBufferMemory

MODEL = "llama3.1"  # Change this to whatever model you have pulled in Ollama

## 2. Connect to Local Ollama LLM

Ollama runs a local server (default: `http://localhost:11434`). Let's verify it's running and check available models.

In [ ]:
# Check which models are available locally
models = ollama.list()
print("Available Ollama models:")
for model in models['models']:
    print(f"  - {model['name']} ({model['size'] / 1e9:.1f} GB)")

# Quick test to verify the model responds
response = ollama.chat(model=MODEL, messages=[
    {"role": "user", "content": "Say hello in one sentence."}
])
print(f"\nTest response from {MODEL}:")
print(response['message']['content'])

## 3. Basic LLM Interaction

Before building agents, let's understand how to interact with the LLM directly. There are two approaches:
1. **Ollama Python client** - simple and direct
2. **LangChain ChatOllama** - provides abstractions for building chains and agents

In [ ]:
# Approach 1: Direct Ollama client - multi-turn conversation
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": "What is the capital of France?"}
]

response = ollama.chat(model=MODEL, messages=messages)
print("User: What is the capital of France?")
print(f"AI: {response['message']['content']}")

# Continue the conversation
messages.append(response['message'])
messages.append({"role": "user", "content": "What is its population?"})

response = ollama.chat(model=MODEL, messages=messages)
print(f"\nUser: What is its population?")
print(f"AI: {response['message']['content']}")

In [ ]:
# Approach 2: LangChain ChatOllama - provides a richer interface
llm = ChatOllama(model=MODEL, temperature=0)

# Simple invocation
response = llm.invoke([
    SystemMessage(content="You are a helpful assistant. Be concise."),
    HumanMessage(content="Explain what an AI agent is in 2 sentences.")
])
print("What is an AI agent?")
print(response.content)

## 4. Build a Simple Agent with Tools

The key difference between a chatbot and an **agent** is **tool use**. An agent can decide to call external functions (tools) to gather information or perform actions.

Let's define some simple tools and build an agent that decides when to use them.

In [ ]:
# Define tools that our agent can use

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Use this for any math calculations.
    Input should be a valid Python math expression like '2 + 2' or '(5 * 3) / 2'."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"

@tool
def search_knowledge(query: str) -> str:
    """Search for factual information. Use this when you need to look up facts,
    definitions, or current information about a topic."""
    # Mock knowledge base - in production, this could call a real search API
    knowledge = {
        "python": "Python is a high-level programming language created by Guido van Rossum in 1991.",
        "ollama": "Ollama is a tool for running large language models locally on your machine.",
        "langchain": "LangChain is a framework for building applications powered by language models.",
        "agent": "An AI agent is a system that uses an LLM to reason and take actions autonomously.",
        "react": "ReAct is a prompting pattern combining Reasoning and Acting for LLM agents.",
    }
    query_lower = query.lower()
    for key, value in knowledge.items():
        if key in query_lower:
            return value
    return f"No specific information found for '{query}'. Try a different search term."

@tool
def get_current_time() -> str:
    """Get the current date and time. Use this when asked about the current time or date."""
    from datetime import datetime
    return f"Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

tools = [calculator, search_knowledge, get_current_time]
print("Defined tools:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

In [ ]:
# Bind tools to the LLM - this tells the model about available tools
llm_with_tools = llm.bind_tools(tools)

# Test: the model decides whether to use a tool or respond directly
response = llm_with_tools.invoke("What is 245 * 38?")
print("Query: What is 245 * 38?")
print(f"Tool calls: {response.tool_calls}")
print(f"Content: {response.content}")

print("\n---")

response2 = llm_with_tools.invoke("What is Ollama?")
print("\nQuery: What is Ollama?")
print(f"Tool calls: {response2.tool_calls}")
print(f"Content: {response2.content}")

## 5. Implement Agent Loop with Reasoning (ReAct Pattern)

The **ReAct** (Reasoning + Acting) pattern is the core of agentic AI. The agent follows this loop:

```
1. THOUGHT: Reason about what to do next
2. ACTION: Choose a tool and provide input
3. OBSERVATION: See the tool's result
4. REPEAT until the task is complete
5. FINAL ANSWER: Respond to the user
```

Let's implement this from scratch to understand how it works under the hood.

In [ ]:
# Build a ReAct agent from scratch using raw Ollama calls

REACT_SYSTEM_PROMPT = """You are an AI assistant that can use tools to help answer questions.

You have access to the following tools:
- calculator(expression): Evaluate math expressions. Input: a Python math expression string.
- search_knowledge(query): Look up factual information. Input: a search query string.
- get_current_time(): Get current date and time. No input needed.

To use a tool, respond in EXACTLY this format:
THOUGHT: [your reasoning about what to do]
ACTION: [tool_name]
ACTION_INPUT: [input to the tool]

When you have enough information to answer, respond with:
THOUGHT: [your final reasoning]
FINAL_ANSWER: [your complete answer to the user]

Always start with a THOUGHT. Never skip the reasoning step."""


def execute_tool(tool_name: str, tool_input: str) -> str:
    """Execute a tool by name and return the result."""
    tool_map = {
        "calculator": calculator,
        "search_knowledge": search_knowledge,
        "get_current_time": get_current_time,
    }
    if tool_name in tool_map:
        return tool_map[tool_name].invoke(tool_input)
    return f"Error: Unknown tool '{tool_name}'"


def parse_agent_response(text: str):
    """Parse the agent's response to extract thought, action, and final answer."""
    lines = text.strip().split("\n")
    thought = ""
    action = ""
    action_input = ""
    final_answer = ""

    for line in lines:
        if line.startswith("THOUGHT:"):
            thought = line[len("THOUGHT:"):].strip()
        elif line.startswith("ACTION:"):
            action = line[len("ACTION:"):].strip()
        elif line.startswith("ACTION_INPUT:"):
            action_input = line[len("ACTION_INPUT:"):].strip()
        elif line.startswith("FINAL_ANSWER:"):
            final_answer = line[len("FINAL_ANSWER:"):].strip()

    return thought, action, action_input, final_answer


def run_agent(user_query: str, max_iterations: int = 5, verbose: bool = True):
    """Run the ReAct agent loop."""
    messages = [
        {"role": "system", "content": REACT_SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    if verbose:
        print(f"🧑 User: {user_query}\n")

    for i in range(max_iterations):
        response = ollama.chat(model=MODEL, messages=messages)
        assistant_text = response['message']['content']

        thought, action, action_input, final_answer = parse_agent_response(assistant_text)

        if verbose:
            print(f"--- Iteration {i+1} ---")
            print(f"💭 Thought: {thought}")

        if final_answer:
            if verbose:
                print(f"✅ Final Answer: {final_answer}")
            return final_answer

        if action:
            if verbose:
                print(f"🔧 Action: {action}({action_input})")

            observation = execute_tool(action, action_input)

            if verbose:
                print(f"👁️ Observation: {observation}\n")

            # Add the assistant's response and the observation to the conversation
            messages.append({"role": "assistant", "content": assistant_text})
            messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})
        else:
            # If no action and no final answer, the model might have just responded directly
            if verbose:
                print(f"💬 Response: {assistant_text}")
            return assistant_text

    return "Agent reached maximum iterations without a final answer."

In [ ]:
# Test the agent with different types of queries

# Query requiring calculation
print("=" * 60)
result = run_agent("What is 15% of 2450?")

print("\n" + "=" * 60)
# Query requiring knowledge lookup
result = run_agent("What is LangChain and how does it relate to AI agents?")

print("\n" + "=" * 60)
# Query requiring time
result = run_agent("What time is it right now?")

## 6. Add Memory to the Agent

A stateless agent forgets everything between queries. By adding **memory**, the agent can:
- Reference previous answers
- Build on earlier reasoning
- Maintain a coherent multi-turn conversation

We'll implement two types of memory:
1. **Full conversation history** - simple but grows unbounded
2. **Summarized memory** - condenses past interactions to save context window space

In [ ]:
class AgentWithMemory:
    """An agent that maintains conversation history across interactions."""

    def __init__(self, model: str = MODEL):
        self.model = model
        self.conversation_history = []
        self.system_prompt = REACT_SYSTEM_PROMPT

    def reset(self):
        """Clear the agent's memory."""
        self.conversation_history = []
        print("🧹 Memory cleared.")

    def get_memory_summary(self) -> str:
        """Get a summary of past interactions."""
        if not self.conversation_history:
            return "No previous interactions."
        summary_lines = []
        for entry in self.conversation_history:
            summary_lines.append(f"Q: {entry['query']}")
            summary_lines.append(f"A: {entry['answer'][:100]}...")
        return "\n".join(summary_lines)

    def run(self, user_query: str, verbose: bool = True) -> str:
        """Run the agent with memory context."""
        # Build context from memory
        memory_context = ""
        if self.conversation_history:
            memory_context = f"\n\nPrevious conversation context:\n{self.get_memory_summary()}\n"

        messages = [
            {"role": "system", "content": self.system_prompt + memory_context},
            {"role": "user", "content": user_query}
        ]

        if verbose:
            print(f"🧑 User: {user_query}")
            if self.conversation_history:
                print(f"📝 Memory: {len(self.conversation_history)} previous interactions")
            print()

        for i in range(5):
            response = ollama.chat(model=self.model, messages=messages)
            assistant_text = response['message']['content']
            thought, action, action_input, final_answer = parse_agent_response(assistant_text)

            if verbose:
                print(f"💭 Thought: {thought}")

            if final_answer:
                if verbose:
                    print(f"✅ Final Answer: {final_answer}\n")
                # Store in memory
                self.conversation_history.append({
                    "query": user_query,
                    "answer": final_answer
                })
                return final_answer

            if action:
                observation = execute_tool(action, action_input)
                if verbose:
                    print(f"🔧 Action: {action}({action_input})")
                    print(f"👁️ Observation: {observation}\n")
                messages.append({"role": "assistant", "content": assistant_text})
                messages.append({"role": "user", "content": f"OBSERVATION: {observation}"})
            else:
                self.conversation_history.append({
                    "query": user_query,
                    "answer": assistant_text
                })
                return assistant_text

        return "Max iterations reached."


# Create an agent with memory
agent = AgentWithMemory()
print("Agent with memory created! Let's have a multi-turn conversation.\n")

In [ ]:
# Demonstrate memory - the agent remembers previous interactions

# First question
agent.run("What is 25 * 4?")

# Follow-up that references the previous answer
agent.run("Now multiply that result by 3")

# Ask about conversation history
agent.run("What calculations have we done so far?")

## 7. Multi-Step Task Execution

Real agentic AI shines with **complex tasks** that require:
- Breaking down the problem into sub-tasks
- Using multiple tools in sequence
- Synthesizing results into a coherent answer

This demonstrates the full power of the agent loop.

In [ ]:
# Multi-step task: requires planning, multiple tool calls, and synthesis

PLANNING_SYSTEM_PROMPT = """You are an AI assistant that solves complex problems by breaking them into steps.

You have access to these tools:
- calculator(expression): Evaluate math expressions
- search_knowledge(query): Look up factual information  
- get_current_time(): Get current date and time

For complex tasks, first create a plan, then execute each step.

Respond in this format:

THOUGHT: [your reasoning - what do you need to do?]
ACTION: [tool_name]
ACTION_INPUT: [input]

OR when you have the final answer:

THOUGHT: [synthesis of all information gathered]
FINAL_ANSWER: [complete answer addressing all parts of the question]

Be thorough - use multiple tools if needed. Always think before acting."""


def run_multi_step_agent(user_query: str, verbose: bool = True) -> str:
    """Agent that handles complex multi-step tasks."""
    messages = [
        {"role": "system", "content": PLANNING_SYSTEM_PROMPT},
        {"role": "user", "content": user_query}
    ]

    if verbose:
        print(f"🧑 Complex Task: {user_query}\n")
        print("=" * 60)

    all_observations = []

    for i in range(8):  # Allow more iterations for complex tasks
        response = ollama.chat(model=MODEL, messages=messages)
        assistant_text = response['message']['content']
        thought, action, action_input, final_answer = parse_agent_response(assistant_text)

        if verbose:
            print(f"\n📍 Step {i+1}:")
            print(f"   💭 Thought: {thought}")

        if final_answer:
            if verbose:
                print(f"\n{'=' * 60}")
                print(f"✅ FINAL ANSWER:\n{final_answer}")
            return final_answer

        if action:
            observation = execute_tool(action, action_input)
            all_observations.append(f"{action}({action_input}) → {observation}")
            if verbose:
                print(f"   🔧 Action: {action}({action_input})")
                print(f"   👁️ Result: {observation}")
            messages.append({"role": "assistant", "content": assistant_text})
            messages.append({"role": "user", "content": f"OBSERVATION: {observation}\n\nContinue with the next step or provide your FINAL_ANSWER."})
        else:
            if verbose:
                print(f"\n💬 Direct response:\n{assistant_text}")
            return assistant_text

    return "Task incomplete - reached maximum iterations."


# Run a complex multi-step task
result = run_multi_step_agent(
    "I need to understand AI agents. First, look up what an AI agent is. "
    "Then look up what the ReAct pattern is. "
    "Finally, calculate how many minutes are in 3.5 hours and tell me "
    "that's roughly how long it takes to build a basic agent from scratch. "
    "Summarize everything together."
)

## 8. Using LangChain for Agents (Higher-Level Approach)

LangChain provides pre-built agent patterns so you don't have to implement the loop yourself. This uses the same Ollama model but with LangChain's agent framework.

In [ ]:
# LangChain agent with tool calling - a more production-ready approach
from langchain_core.messages import ToolMessage

llm_agent = ChatOllama(model=MODEL, temperature=0).bind_tools(tools)

def run_langchain_agent(query: str, verbose: bool = True):
    """Run a LangChain-style agent with tool calling loop."""
    messages = [
        SystemMessage(content="You are a helpful assistant. Use tools when needed to answer accurately."),
        HumanMessage(content=query)
    ]

    if verbose:
        print(f"🧑 Query: {query}\n")

    for i in range(5):
        response = llm_agent.invoke(messages)
        messages.append(response)

        # If no tool calls, we have our final answer
        if not response.tool_calls:
            if verbose:
                print(f"✅ Answer: {response.content}")
            return response.content

        # Execute each tool call
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            if verbose:
                print(f"🔧 Calling: {tool_name}({tool_args})")

            # Execute the tool
            tool_map = {t.name: t for t in tools}
            if tool_name in tool_map:
                result = tool_map[tool_name].invoke(tool_args)
            else:
                result = f"Unknown tool: {tool_name}"

            if verbose:
                print(f"   → {result}")

            # Add tool result to messages
            messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

    return "Max iterations reached."

# Test the LangChain agent
run_langchain_agent("What is 1337 * 42?")
print()
run_langchain_agent("What is Python and what time is it?")

## Key Takeaways

| Concept | Description |
|---------|-------------|
| **LLM** | The "brain" - generates text, reasons about problems |
| **Tools** | Functions the agent can call (APIs, calculators, search, etc.) |
| **ReAct Loop** | Think → Act → Observe → Repeat cycle |
| **Memory** | Conversation history that persists across turns |
| **Planning** | Breaking complex tasks into manageable steps |

### Architecture of an Agent

```
User Query → [Agent Loop] → Final Answer
                  ↓
         ┌───────────────┐
         │   LLM Thinks  │ ← Reasoning
         │   (THOUGHT)   │
         └───────┬───────┘
                 ↓
         ┌───────────────┐
         │ Select Action  │ ← Tool Selection
         │   (ACTION)    │
         └───────┬───────┘
                 ↓
         ┌───────────────┐
         │ Execute Tool   │ ← Tool Execution
         │(OBSERVATION)  │
         └───────┬───────┘
                 ↓
         ┌───────────────┐
         │ Loop or Final  │ ← Decision
         │   Answer?     │
         └───────────────┘
```

### Next Steps
- Try adding more tools (file reader, web scraper, database query)
- Experiment with different Ollama models (mistral, codellama, phi3)
- Explore **LangGraph** for more complex multi-agent workflows
- Add structured output parsing for more reliable tool calling